In [2]:
import subprocess
import sys

def install_if_missing(module_name, pip_name=None):
    try:
        __import__(module_name)
    except ImportError:
        if pip_name is None:
            pip_name = module_name
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])

# Lista bibliotek: moduł_import -> paczka_pip
required_packages = {
    "os": None,
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "open3d": "open3d",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
    "plotly": "plotly"
}

for module_name, pip_name in required_packages.items():
    install_if_missing(module_name, pip_name)


In [ ]:
#---------------------- Libraries ----------------------#
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import open3d as o3d
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from scipy.spatial import ConvexHull
import plotly.express as px
import plotly.graph_objects as go
import matplotlib

#---------------------- Read Data ----------------------#
folder_input = "./data_analiza_raw/raw"
folder_output = "./data_analiza_raw/prod"
os.makedirs(folder_output, exist_ok=True)

file_list = sorted([f for f in os.listdir(folder_input) if f.endswith(".csv")])
if not file_list:
    raise FileNotFoundError("Brak plików CSV w folderze 'raw'.")

warstwa_z = 5
all_data = []

def filter_data(df):
    return df[df["distance"] > 0]

#---------------------- Processing Data ----------------------#
for i, file_name in enumerate(file_list):
    file_path = os.path.join(folder_input, file_name)
    try:
        data = pd.read_csv(file_path, header=None, delimiter=r"\s+", skiprows=3)
        if data.shape[1] < 2:
            raise ValueError(f"Plik {file_name} nie ma wymaganych kolumn.")
    except Exception as e:
        print(f"Błąd podczas przetwarzania {file_name}: {e}")
        continue

    data = data.iloc[:, :2]
    data.columns = ['angle', 'distance']
    data = filter_data(data)

    data["x"] = data["distance"] * np.cos(np.radians(data["angle"]))
    data["y"] = data["distance"] * np.sin(np.radians(data["angle"]))
    data["z"] = i * warstwa_z

    processed_file_path = os.path.join(folder_output, f"new_data_{i+1}.csv")
    data.to_csv(processed_file_path, index=False)

    cartesian_file_path = os.path.join(folder_output, f"cartesian_data_{i+1}.csv")
    data[["x", "y", "z"]].to_csv(cartesian_file_path, index=False)

    all_data.append(data[["x", "y", "z"]])

final_data = pd.concat(all_data, ignore_index=True)

#---------------------- Scale Data & Clustering ----------------------#
scaler = StandardScaler()
X_scaled = scaler.fit_transform(final_data[["x", "y", "z"]])

dbscan = DBSCAN(eps=0.4, min_samples=15)
clusters = dbscan.fit_predict(X_scaled)

final_data["cluster"] = clusters

#---------------------- Open3D - Chmura punktów z dopasowaną podłogą i sufitem ----------------------#

# --- Dane z chmury punktów ---
points = final_data[["x", "y", "z"]].values.astype(np.float64)
clusters = final_data["cluster"].values
unique_clusters = np.unique(clusters)

# --- Kolorowanie klastrów ---
colormap = matplotlib.colormaps.get_cmap("tab20")
colors = np.zeros((len(clusters), 3))
for idx, cl in enumerate(unique_clusters):
    if cl == -1:
        colors[clusters == cl] = [0.5, 0.5, 0.5]
    else:
        colors[clusters == cl] = colormap(idx % 20)[:3]

# --- Chmura punktów ---
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd.colors = o3d.utility.Vector3dVector(colors)

# --- Funkcja tworząca dopasowaną siatkę wielokąta ---
def create_convex_mesh(points_slice, z_value, color):
    xy = points_slice[:, :2]
    hull = ConvexHull(xy)
    hull_points = xy[hull.vertices]
    vertices = np.hstack((hull_points, np.full((len(hull_points), 1), z_value)))
    
    # Wierzchołki
    mesh = o3d.geometry.TriangleMesh()
    mesh.vertices = o3d.utility.Vector3dVector(vertices)
    
    # Trójkąty przód
    triangles = [[0, i, i + 1] for i in range(1, len(vertices) - 1)]
    
    # Trójkąty tył (odwrócone)
    triangles += [[0, i + 1, i] for i in range(1, len(vertices) - 1)]
    
    mesh.triangles = o3d.utility.Vector3iVector(triangles)
    mesh.paint_uniform_color(color)
    mesh.compute_vertex_normals()
    return mesh


# --- Wyznaczenie najniższych i najwyższych punktów ---
z_min, z_max = points[:, 2].min(), points[:, 2].max()
tolerance = 1.0  # zakres wysokości do uznania za podłogę/sufit

floor_points = points[points[:, 2] < z_min + tolerance]
ceiling_points = points[points[:, 2] > z_max - tolerance]

# --- Tworzenie podłogi i sufitu dopasowanych do kształtu ---
floor = create_convex_mesh(floor_points, z_min - 0.05, [0.5, 0.6, 0.2])
ceiling = create_convex_mesh(ceiling_points, z_max + 0.05, [0.5, 0.5, 0.9])

# --- Ramka współrzędnych ---
frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=50, origin=[0, 0, 0])

# --- Widoczność ---
visibility = {"floor": True, "ceiling": True}

# --- Visualizer ---
vis = o3d.visualization.VisualizerWithKeyCallback()
vis.create_window(window_name="Open3D - Dopasowana podłoga i sufit")
vis.add_geometry(pcd)
vis.add_geometry(frame)
vis.add_geometry(floor)
vis.add_geometry(ceiling)

def toggle_floor(vis):
    if visibility["floor"]:
        vis.remove_geometry(floor)
    else:
        vis.add_geometry(floor)
    visibility["floor"] = not visibility["floor"]
    return False

def toggle_ceiling(vis):
    if visibility["ceiling"]:
        vis.remove_geometry(ceiling)
    else:
        vis.add_geometry(ceiling)
    visibility["ceiling"] = not visibility["ceiling"]
    return False

vis.register_key_callback(ord("F"), toggle_floor)
vis.register_key_callback(ord("C"), toggle_ceiling)

vis.run()
vis.destroy_window()


#---------------------- Visualization 3D Plotly ----------------------#
fig = px.scatter_3d(final_data, x="x", y="y", z="z", color=final_data["cluster"].astype(str),
                    title="DBSCAN - Klasteryzacja 3D", opacity=0.6,
                    labels={"cluster": "Klaster"})
fig.update_traces(marker=dict(size=3))
fig.show(renderer="browser")

#---------------------- Convex Hull - Browser ----------------------#
points_3d = final_data[["x", "y", "z"]].values
hull_3d = ConvexHull(points_3d)

hull_mesh = go.Mesh3d(
    x=points_3d[:, 0],
    y=points_3d[:, 1],
    z=points_3d[:, 2],
    i=hull_3d.simplices[:, 0],
    j=hull_3d.simplices[:, 1],
    k=hull_3d.simplices[:, 2],
    opacity=0.5,
    color='lightblue',
    name='Convex Hull'
)

hull_fig = go.Figure()
hull_fig.add_trace(go.Scatter3d(
    x=final_data["x"], y=final_data["y"], z=final_data["z"],
    mode='markers',
    marker=dict(size=3, color=final_data["cluster"], colorscale='Viridis', opacity=0.5),
    name='LiDAR Points'
))
hull_fig.add_trace(hull_mesh)
hull_fig.update_layout(title="3D Convex Hull - LiDAR Points",
                       scene=dict(xaxis_title='X [mm]',
                                  yaxis_title='Y [mm]',
                                  zaxis_title='Z [mm]'))
hull_fig.show(renderer="browser")

#---------------------- 2D Scatter Plot ----------------------#
plt.figure(figsize=(8, 5))
plt.scatter(final_data["x"], final_data["y"], c=final_data["z"], cmap='viridis', s=1)
plt.xlabel("X [mm]")
plt.ylabel("Y [mm]")
plt.title("Rzut 2D punktów LiDAR (kolor: wysokość Z)")
plt.colorbar(label="Z [mm]")
plt.axis("equal")
plt.show()

#---------------------- Data scatter ----------------------#
print("Rozrzut danych:")
print("X:", final_data["x"].min(), "->", final_data["x"].max())
print("Y:", final_data["y"].min(), "->", final_data["y"].max())
print("Z:", final_data["z"].min(), "->", final_data["z"].max())


In [ ]:
import numpy as np
import open3d as o3d
import matplotlib.pyplot as plt
import matplotlib

points = final_data[["x", "y", "z"]].values.astype(np.float64)
z_min, z_max = points[:, 2].min(), points[:, 2].max()
num_layers = 40

layers = np.clip(((points[:, 2] - z_min) / (z_max - z_min) * num_layers).astype(int), 0, num_layers - 1)
colors = matplotlib.colormaps["tab20"](layers % 20)[:, :3]

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)
pcd.colors = o3d.utility.Vector3dVector(colors)

origin = points.min(axis=0)
frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=50, origin=[0, 0, 0])

vis = o3d.visualization.VisualizerWithEditing()
vis.create_window(window_name="Wybierz 2 punkty", width=800, height=600)
vis.add_geometry(pcd)
vis.add_geometry(frame)
vis.run()
picked = vis.get_picked_points()
vis.destroy_window()

print("Wybrane indeksy:", picked)
if len(picked) == 2:
    dist = np.linalg.norm(points[picked[0]] - points[picked[1]])
    print(f"Odległość między punktami: {dist:.2f} mm")
else:
    print("Musisz wybrać dokładnie 2 punkty.")
